# ¿Qué palabra falta? BETO vs Tulio vs Patana

Los tres son modelos BERT del DCC de la Universidad de Chile:

| Modelo | Qué leyó |
| --- | --- |
| **BETO** | Español general |
| **Tulio** | BETO + textos de Chile (noticias, web, reclamos, tweets) y libros en español |
| **Patana** | Otra variante ajustada a español de Chile |

Estos modelos **no conversan**. Lo que saben hacer de fábrica es adivinar una palabra tapada con `[MASK]`. Si le damos la misma frase a los tres, se ve qué aprendió cada uno.

No necesita GPU, aunque con GPU va más rápido.

In [ ]:
!pip install -q transformers

In [ ]:
import pandas as pd
from transformers import pipeline
import torch

MODELOS = {
    "BETO":   "dccuchile/bert-base-spanish-wwm-cased",
    "Tulio":  "dccuchile/tulio-chilean-spanish-bert",
    "Patana": "dccuchile/patana-chilean-spanish-bert",
}

dispositivo = 0 if torch.cuda.is_available() else -1
completadores = {nombre: pipeline("fill-mask", model=ruta, device=dispositivo)
                 for nombre, ruta in MODELOS.items()}
print("✅ Modelos cargados:", ", ".join(completadores))

## Comparar los tres modelos

Para cada frase se muestran las 5 palabras más probables de cada modelo. Escribe `[MASK]` donde va la palabra tapada (solo una por frase).

In [ ]:
def comparar(frase, top=5):
    columnas = {}
    for nombre, pipe in completadores.items():
        texto = frase.replace("[MASK]", pipe.tokenizer.mask_token)
        columnas[nombre] = [f"{r['token_str'].strip()} ({r['score']:.0%})" for r in pipe(texto, top_k=top)]
    print(f"\n📝 {frase}")
    display(pd.DataFrame(columnas, index=range(1, top + 1)))

FRASES = [
    "El carrete estuvo muy [MASK].",
    "Qué [MASK] la película, me quedé dormido.",
    "Mañana me levanto temprano porque tengo que ir a la [MASK].",
    "Para el dieciocho comimos [MASK] con pebre.",
    "Pasé a la panadería a comprar [MASK] para la once.",
    "Ese profe explica súper bien, es muy [MASK].",
]

for f in FRASES:
    comparar(f)

## Prueba en vivo

Pide frases al curso. Funciona mejor si la palabra tapada es típica de Chile (comida, garabatos suaves, expresiones, lugares).

In [ ]:
comparar("Oye, ¿[MASK] lo que te dije?")

## Para discutir

- ¿En qué frases Tulio o Patana proponen palabras más chilenas que BETO? ¿En cuáles no hay diferencia?
- Los porcentajes son la **probabilidad** que el modelo le asigna a cada palabra. ¿Están muy seguros o dudan entre varias opciones?
- Tulio se entrenó con tweets y reclamos sin filtrar. ¿Qué riesgos tiene eso? Prueben frases sobre grupos de personas y comenten si aparecen sesgos.

**Siguiente paso:** estos modelos se pueden usar como punto de partida para un fine-tuning. En el notebook de conceptos, Parte 1, basta cambiar `BERT = "dccuchile/bert-base-spanish-wwm-cased"` por `"dccuchile/tulio-chilean-spanish-bert"` para entrenar el clasificador de sentimientos a partir de Tulio.